[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch05-nn-computation/03-ba-loi-tinh-tren-mot-chu-so.ipynb)

# Ba lối tính trên một chữ số: quy tắc, HOG + SVM, và mạng nơ-ron

Chương 5 theo một chữ số viết tay 28×28 qua ba lối tính. Lối quy tắc tốn khoảng 100 phép so
sánh, lối học máy cổ điển với HOG và SVM khoảng 8,000 phép, còn mạng 784-128-64-10 tốn 109,184
MAC. Chương dừng ở số phép. Sổ tay này dựng cả ba, đếm phép của từng lối, rồi đo thứ mà con số
của chương không nói: mỗi lối phân loại đúng bao nhiêu, và chạy mất bao lâu trên CPU của bạn.

**Bạn sẽ làm:**
1. viết một bộ quy tắc đúng 100 phép so sánh, và đếm từng phép;
2. tự tính HOG với lưới 7×7 ô, 9 ngăn, ra vectơ 441 phần tử, rồi huấn luyện một SVM tuyến tính;
3. huấn luyện mạng 784-128-64-10 và đặt ba lối cạnh nhau;
4. kiểm hai lời cảnh báo của chương về quy mô: dựng mốc đơn giản trước, và lợi ích giảm dần.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import pandas as pd
import torch.nn.functional as F
import torchvision

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # mỗi lối tính một màu cố định

tap_huan_luyen = torchvision.datasets.MNIST(DATA, train=True, download=True)
tap_kiem_tra = torchvision.datasets.MNIST(DATA, train=False, download=True)
N_HL, N_KT = 10_000, 5_000  # tập con, để cả sổ tay chạy trong vài chục giây
anh_hl, nhan_hl = tap_huan_luyen.data[:N_HL].numpy(), tap_huan_luyen.targets[:N_HL].numpy()
anh_kt, nhan_kt = tap_kiem_tra.data[:N_KT].numpy(), tap_kiem_tra.targets[:N_KT].numpy()

theo_sach("số pixel của một ảnh", anh_hl.shape[1] * anh_hl.shape[2], sach=784, nguon="28×28 = 784 pixel")
print(f"huấn luyện trên {N_HL:,} ảnh, kiểm tra trên {N_KT:,} ảnh, kiểu {anh_hl.dtype}")


def thoi_gian(f, lap=5):
    """Trung vị thời gian của f(), sau một lần chạy khởi động."""
    f()
    do = []
    for _ in range(lap):
        t0 = time.perf_counter()
        f()
        do.append(time.perf_counter() - t0)
    return float(np.median(do))

## 1. Lối thứ nhất: quy tắc viết tay

Chương mô tả một bộ phân loại quy tắc minh hoạ: so cường độ pixel với ngưỡng, kiểm nét chữ ở
từng vùng, rồi rẽ nhánh theo kết quả. Nó tốn khoảng 100 phép so sánh trên 784 byte dữ liệu pixel.

Bộ quy tắc của sổ tay này dùng đúng ngân sách đó: với mỗi chữ số, 5 điểm phải có mực và 5 điểm
phải trống, tức 10 phép so sánh cho mỗi chữ số, 100 phép cho cả mười. Chữ số thoả nhiều quy tắc
nhất thắng. Một người viết quy tắc sẽ nhìn ảnh trung bình của từng chữ số rồi chọn điểm; ở đây
máy chọn hộ theo đúng cách người đó nhìn, để khỏi gõ tay 100 toạ độ.

In [ ]:
SO_SANH = sach(100, nguon="~100 phép so sánh", tol=10)
BYTE_ANH = sach(784, nguon="784 byte")
assert anh_kt[0].nbytes == BYTE_ANH  # một ảnh uint8 là đúng 784 byte

MOI_CHU_SO = SO_SANH // 10  # 10 phép so sánh cho mỗi chữ số
NGUONG = 128                # nửa thang xám của uint8

trung_binh = np.stack([anh_hl[nhan_hl == d].reshape(-1, 784).mean(0) for d in range(10)])
diem_muc, diem_trong = [], []
for d in range(10):
    khac = np.delete(trung_binh, d, axis=0).mean(0)
    chenh = trung_binh[d] - khac
    diem_muc.append(np.argsort(-chenh)[: MOI_CHU_SO // 2])   # chữ số d có mực, các chữ số khác thì không
    diem_trong.append(np.argsort(chenh)[: MOI_CHU_SO // 2])  # các chữ số khác có mực, chữ số d thì không
diem_muc, diem_trong = np.array(diem_muc), np.array(diem_trong)


def quy_tac_mot_anh(anh):
    """Phân loại một ảnh, đếm từng phép so sánh."""
    phang, dem = anh.reshape(-1), 0
    diem = []
    for d in range(10):
        thoa = 0
        for i in diem_muc[d]:
            thoa += int(phang[i] > NGUONG)
            dem += 1
        for i in diem_trong[d]:
            thoa += int(phang[i] <= NGUONG)
            dem += 1
        diem.append(thoa)
    tot_nhat = 0
    for d in range(1, 10):  # tìm chữ số có điểm cao nhất: 9 phép so sánh nữa
        dem += 1
        if diem[d] > diem[tot_nhat]:
            tot_nhat = d
    return tot_nhat, dem


def quy_tac(anh):
    """Cùng bộ quy tắc, viết theo vectơ cho cả tập."""
    phang = anh.reshape(len(anh), 784)
    diem = (phang[:, diem_muc] > NGUONG).sum(2) + (phang[:, diem_trong] <= NGUONG).sum(2)
    return diem.argmax(1)


_, so_phep_quy_tac = quy_tac_mot_anh(anh_kt[0])
print(f"một ảnh: {so_phep_quy_tac} phép so sánh ({SO_SANH} kiểm điểm ảnh + 9 để chọn chữ số thắng)")
assert all(quy_tac_mot_anh(a)[0] == p for a, p in zip(anh_kt[:300], quy_tac(anh_kt[:300])))

### Dự đoán

Đoán ngẫu nhiên giữa mười chữ số thì đúng khoảng một phần mười. Bộ quy tắc 100 phép so sánh này
sẽ đúng bao nhiêu trên tập kiểm tra: gần như mọi ảnh, khoảng một nửa, hay chỉ hơn đoán mò một chút?

In [ ]:
du_doan_quy_tac = quy_tac(anh_kt)
dung_quy_tac = (du_doan_quy_tac == nhan_kt).mean()
do_tren_may("độ chính xác của bộ quy tắc", 100 * dung_quy_tac, "%")
for d in range(10):
    print(f"  chữ số {d}: đúng {(du_doan_quy_tac[nhan_kt == d] == d).mean():.0%}")

fig, axes = plt.subplots(2, 5, figsize=(10, 4.4))
for d, ax in enumerate(axes.flat):
    ax.imshow(trung_binh[d].reshape(28, 28), cmap="gray_r")
    r, c = np.divmod(diem_muc[d], 28)
    ax.plot(c, r, "o", ms=7, mfc="none", mec=MAU[1], mew=2)
    r, c = np.divmod(diem_trong[d], 28)
    ax.plot(c, r, "x", ms=7, color=MAU[0], mew=2)
    ax.set_title(f"chữ số {d}", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
fig.suptitle("Ảnh trung bình và quy tắc: o phải có mực, x phải trống", fontsize=11)
plt.tight_layout()
plt.show()

Bộ quy tắc rẻ và dễ đoán: 784 byte, 109 phép so sánh, vừa bộ nhớ đệm của bất kỳ CPU nào. Nhưng
mỗi chữ số viết lệch, nghiêng hay to hơn ảnh trung bình là một trường hợp mà 10 điểm cố định
không lường trước, và muốn sửa thì phải thêm quy tắc. Chương gọi đây là mốc so sánh về tải công
việc (workload), không phải thước đo độ chính xác của một hệ quy tắc thật.

## 2. Lối thứ hai: HOG, rồi một SVM tuyến tính

Thiết kế đặc trưng (feature engineering) biến pixel thô thành một biểu diễn mà thuật toán học dễ
dùng hơn. Histogram hướng gradient, HOG, tìm cạnh nơi độ sáng đổi đột ngột, chia ảnh thành ô, rồi
đo hướng các cạnh trong từng ô. Trong hiện thực minh hoạ của chương:

In [ ]:
O_LUOI = sach(7, nguon="ô 7×7 / 4×4")
O_PIXEL = sach(4, nguon="ô 7×7 / 4×4")
NGAN = sach(9, nguon="9 ngăn")
SO_LOP = sach(10, nguon="10 tích vô hướng")
assert O_LUOI * O_PIXEL == 28
DAI_HOG = O_LUOI * O_LUOI * NGAN
theo_sach("độ dài vectơ đặc trưng HOG", DAI_HOG, sach=441, nguon="441 đặc trưng")
print(f"{O_LUOI}×{O_LUOI} ô, mỗi ô {O_PIXEL}×{O_PIXEL} pixel, {NGAN} ngăn hướng → {DAI_HOG} đặc trưng")

Bốn bước, viết bằng NumPy:

1. gradient theo hai trục bằng hiệu của hai pixel kề hai bên;
2. độ lớn $\sqrt{g_x^2 + g_y^2}$ và hướng $\operatorname{atan2}(g_y, g_x)$, gấp về khoảng 0° đến 180°;
3. chia hướng vào 9 ngăn, mỗi ngăn 20°;
4. cộng độ lớn vào ngăn của nó, theo từng ô 4×4.

In [ ]:
o_cua_pixel = (np.arange(28) // O_PIXEL)
chi_so_o = o_cua_pixel[:, None] * O_LUOI + o_cua_pixel[None, :]  # ô của từng pixel, 28 × 28


def hog(anh):
    x = anh.astype(np.float32) / 255
    gx, gy = np.zeros_like(x), np.zeros_like(x)
    gx[:, :, 1:-1] = x[:, :, 2:] - x[:, :, :-2]                       # bước 1
    gy[:, 1:-1, :] = x[:, 2:, :] - x[:, :-2, :]
    do_lon = np.sqrt(gx ** 2 + gy ** 2)                               # bước 2
    huong = np.rad2deg(np.arctan2(gy, gx)) % 180
    ngan = np.minimum((huong // (180 / NGAN)).astype(int), NGAN - 1)  # bước 3
    H = np.zeros((len(x), DAI_HOG), np.float32)                       # bước 4
    np.add.at(H, (np.arange(len(x))[:, None, None], chi_so_o[None] * NGAN + ngan), do_lon)
    return H


hog_hl, hog_kt = hog(anh_hl), hog(anh_kt)
print("đặc trưng HOG:", hog_hl.shape)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
mau_anh = anh_kt[0]
ax1.imshow(mau_anh, cmap="gray_r")
ax1.set_title(f"ảnh gốc (chữ số {nhan_kt[0]})", fontsize=10)
H0 = hog(mau_anh[None])[0].reshape(O_LUOI, O_LUOI, NGAN)
for i in range(O_LUOI):
    for j in range(O_LUOI):
        tam_y, tam_x = O_PIXEL * i + 1.5, O_PIXEL * j + 1.5
        for b in range(NGAN):
            goc = np.deg2rad((b + 0.5) * 180 / NGAN + 90)  # cạnh vuông góc với gradient
            dai = 1.9 * H0[i, j, b] / H0.max()
            ax2.plot([tam_x - dai * np.cos(goc), tam_x + dai * np.cos(goc)],
                     [tam_y - dai * np.sin(goc), tam_y + dai * np.sin(goc)], color=MAU[0], lw=1.2)
ax2.set_xlim(-0.5, 27.5)
ax2.set_ylim(27.5, -0.5)
ax2.set_aspect("equal")
ax2.set_xticks(np.arange(-0.5, 28, O_PIXEL), [])
ax2.set_yticks(np.arange(-0.5, 28, O_PIXEL), [])
ax2.tick_params(length=0)
ax2.set_title("HOG: 49 ô, mỗi ô 9 hướng cạnh", fontsize=10)
for ax in (ax1,):
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
plt.tight_layout()
plt.show()

Mỗi ô giờ chỉ giữ hướng của các nét đi qua nó, không giữ vị trí chính xác của từng pixel. Đó là
lý do HOG bền trước thay đổi ánh sáng và những dịch chuyển nhỏ.

Bước cuối là một SVM tuyến tính: mười vectơ trọng số, mỗi chữ số một vectơ, và chữ số có tích vô
hướng lớn nhất thắng. Sổ tay huấn luyện nó bằng hàm mất mát hinge, như mọi SVM tuyến tính, với
một chút suy giảm trọng số (weight decay).

In [ ]:
trung_tam, do_lech = hog_hl.mean(0), hog_hl.std(0) + 1e-6
Xh_hl = torch.tensor((hog_hl - trung_tam) / do_lech)
Xh_kt = torch.tensor((hog_kt - trung_tam) / do_lech)
y_hl, y_kt = torch.tensor(nhan_hl).long(), torch.tensor(nhan_kt).long()


def huan_luyen(mo_hinh, X, y, so_epoch, ham_mat_mat, toi_uu, batch=32):
    """Vòng lặp huấn luyện quen thuộc: xáo trộn, chia batch, truyền xuôi, mất mát, truyền ngược, cập nhật."""
    dat_hat_giong()
    for _ in range(so_epoch):
        thu_tu = torch.randperm(len(X))
        for i in range(0, len(X), batch):
            chon = thu_tu[i:i + batch]
            mat_mat = ham_mat_mat(mo_hinh(X[chon]), y[chon])
            toi_uu.zero_grad()
            mat_mat.backward()
            toi_uu.step()
    return mo_hinh


def do_chinh_xac(mo_hinh, X, y):
    with torch.no_grad():
        return (mo_hinh(X).argmax(1) == y).float().mean().item()


dat_hat_giong()
svm = torch.nn.Linear(DAI_HOG, SO_LOP)
t0 = time.perf_counter()
huan_luyen(svm, Xh_hl, y_hl, 10, F.multi_margin_loss,
           torch.optim.SGD(svm.parameters(), lr=0.05, weight_decay=1e-4))
do_tren_may("thời gian huấn luyện SVM", time.perf_counter() - t0, "s")
dung_hog = do_chinh_xac(svm, Xh_kt, y_kt)
do_tren_may("độ chính xác của HOG + SVM", 100 * dung_hog, "%")

### Đếm phép của lối thứ hai

Chương ước lượng lối này tốn khoảng 8,000 phép toán và khoảng 2 KB bộ nhớ làm việc. Đếm phép của
đúng hiện thực vừa viết, cho một ảnh:

In [ ]:
P = 28 * 28
phep_hog = {
    "gradient (2 phép trừ mỗi pixel)": 2 * P,
    "độ lớn (2 bình phương, 1 cộng, 1 căn)": 4 * P,
    "hướng (1 atan2 mỗi pixel)": P,
    "chia ngăn (1 phép chia)": P,
    "cộng vào histogram": P,
    "SVM: 10 tích vô hướng, đếm 1 MAC là 1 phép": SO_LOP * DAI_HOG,
}
for ten, n in phep_hog.items():
    print(f"  {ten:<45} {n:>7,}")
tong_hog = sum(phep_hog.values())
HOG_SACH = sach(8000, nguon="~8.000 phép", tol=500)
print(f"tổng sổ tay đếm: {tong_hog:,} phép · ước lượng của chương: khoảng {HOG_SACH:,}")
print(f"tỉ lệ: {tong_hog / HOG_SACH:.1f}")

byte_dac_trung = DAI_HOG * 4
byte_svm = sum(p.numel() for p in svm.parameters()) * 4
print(f"vectơ đặc trưng FP32: {byte_dac_trung:,} byte · trọng số SVM: {byte_svm:,} byte")

Cách đếm của sổ tay ra lớn hơn ước lượng của chương, nhưng cùng bậc độ lớn: vài nghìn phép, chứ
không phải vài trăm hay vài trăm nghìn. Chênh lệch nằm ở quy ước đếm: một MAC là một hay hai phép,
một atan2 là một phép hay vài chục lệnh máy. Chương nói rõ kích thước bộ mô tả, số lớp và cách hiện
thực đều làm đổi chi phí này.

Với bộ nhớ, riêng vectơ đặc trưng FP32 đã gần 2 KB, còn trọng số SVM thì lớn hơn thế nhiều lần.
Chương không nói 2 KB gồm những khoản nào, nên ở đây không so khớp con số đó.

Hai tỉ lệ của chương thì tính lại được từ chính các con số của chương:

In [ ]:
theo_sach("HOG + SVM so với quy tắc", HOG_SACH / SO_SANH, sach=80, nguon="80× · ô 7×7")
print(f"{HOG_SACH:,} phép / {SO_SANH} phép = {HOG_SACH / SO_SANH:.0f}×")

## 3. Lối thứ ba: mạng 784-128-64-10

Mạng không cần bộ trích đặc trưng nào: nó nhận thẳng 784 pixel, chuẩn hoá về khoảng 0 đến 1, và
tự học biểu diễn từ dữ liệu.

In [ ]:
def mlp(rong=(128, 64)):
    lop, vao = [], 784
    for r in rong:
        lop += [torch.nn.Linear(vao, r), torch.nn.ReLU()]
        vao = r
    return torch.nn.Sequential(*lop, torch.nn.Linear(vao, 10))


X_hl = torch.tensor(anh_hl.reshape(-1, 784) / 255, dtype=torch.float32)
X_kt = torch.tensor(anh_kt.reshape(-1, 784) / 255, dtype=torch.float32)

dat_hat_giong()
mang = mlp()
t0 = time.perf_counter()
huan_luyen(mang, X_hl, y_hl, 5, F.cross_entropy, torch.optim.Adam(mang.parameters(), lr=1e-3))
do_tren_may("thời gian huấn luyện mạng, 5 epoch", time.perf_counter() - t0, "s")
dung_mang = do_chinh_xac(mang, X_kt, y_kt)
do_tren_may("độ chính xác của mạng 784-128-64-10", 100 * dung_mang, "%")

mac_mang = sum(m.in_features * m.out_features for m in mang if isinstance(m, torch.nn.Linear))
tham_so = sum(p.numel() for p in mang.parameters())
theo_sach("MAC mỗi ảnh", mac_mang, sach=109_184, nguon="109.184 MAC")
theo_sach("mạng so với quy tắc", mac_mang / SO_SANH, sach=1091.8, nguon="1.091,8×")
theo_sach("tham số FP32 (KB)", tham_so * 4 / 1000, sach=438, nguon="438 KB (FP32)")
print(f"{tham_so:,} tham số, {mac_mang:,} MAC mỗi ảnh")

### Dự đoán

Mạng tốn hơn quy tắc khoảng một nghìn lần số phép, và hơn HOG + SVM khoảng mười lần. Nó có chắc
chính xác nhất không? Còn thời gian mỗi ảnh trên CPU thì tăng theo đúng tỉ lệ số phép không?

In [ ]:
t_quy_tac = thoi_gian(lambda: quy_tac(anh_kt)) / N_KT
t_quy_tac_python = thoi_gian(lambda: [quy_tac_mot_anh(a) for a in anh_kt[:500]], lap=3) / 500
t_hog = thoi_gian(lambda: svm(torch.tensor((hog(anh_kt) - trung_tam) / do_lech)).argmax(1)) / N_KT
with torch.no_grad():
    t_mang = thoi_gian(lambda: mang(torch.tensor(anh_kt.reshape(-1, 784) / 255, dtype=torch.float32)).argmax(1)) / N_KT

bang = pd.DataFrame({
    "lối": ["quy tắc", "HOG + SVM", "mạng 784-128-64-10"],
    "phép mỗi ảnh (chương)": [SO_SANH, HOG_SACH, 109_184],
    "phép mỗi ảnh (sổ tay đếm)": [so_phep_quy_tac, tong_hog, mac_mang],
    "giá trị lưu trong mô hình": [diem_muc.size + diem_trong.size, byte_svm // 4, tham_so],
    "độ chính xác, đo (%)": [100 * dung_quy_tac, 100 * dung_hog, 100 * dung_mang],
    "µs mỗi ảnh, đo": [1e6 * t_quy_tac, 1e6 * t_hog, 1e6 * t_mang],
})
for ten, t in zip(bang["lối"], bang["µs mỗi ảnh, đo"]):
    do_tren_may(f"thời gian mỗi ảnh, {ten}", t, "µs")
do_tren_may("thời gian mỗi ảnh, quy tắc viết bằng vòng lặp Python", 1e6 * t_quy_tac_python, "µs")
bang.round({"độ chính xác, đo (%)": 1, "µs mỗi ảnh, đo": 2})

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
phep_sach = bang["phép mỗi ảnh (chương)"].to_numpy(float)
ax1.bar(bang["lối"], phep_sach, color=MAU[:3], width=0.6)
ax1.set_yscale("log")
for i, (v, ti_le) in enumerate(zip(phep_sach, ["", "80× quy tắc", "1,091.8× quy tắc"])):
    ax1.text(i, v * 1.3, f"{v:,.0f}", ha="center", fontsize=9)
    if ti_le:
        ax1.text(i, v * 4, ti_le, ha="center", fontsize=9, color="0.3")
ax1.set_ylim(30, 1e6)
ax1.set_ylabel("phép mỗi ảnh (thang log)")
ax1.set_title("Số phép theo chương")

for i in range(3):
    ax2.plot(bang["phép mỗi ảnh (sổ tay đếm)"][i], bang["độ chính xác, đo (%)"][i], "o", ms=10, color=MAU[i])
    ax2.annotate(bang["lối"][i], (bang["phép mỗi ảnh (sổ tay đếm)"][i], bang["độ chính xác, đo (%)"][i]),
                 xytext=(0, -20), textcoords="offset points", ha="center", fontsize=9)
ax2.set_xscale("log")
ax2.set_xlim(30, 1e6)
ax2.set_ylim(0, 105)
ax2.axhline(10, color="0.6", lw=0.8, ls="--")
ax2.text(40, 12, "đoán mò: 10 %", fontsize=8, color="0.4")
ax2.set_xlabel("phép mỗi ảnh, sổ tay đếm (thang log)")
ax2.set_ylabel("độ chính xác trên tập kiểm tra (%)")
ax2.set_title("Đo trên máy này")
plt.tight_layout()
plt.show()

Đọc bảng theo hai chiều.

Về tải công việc, đúng như chương nói: mỗi bước chuyển nhân số phép lên nhiều lần, và đổi luôn
dạng của công việc, từ so sánh rẽ nhánh, sang phép toán có cấu trúc trên vectơ đặc trưng, sang phép
nhân ma trận dày đặc. Tham số của mạng, khoảng 438 KB, đã vượt phần lớn bộ nhớ đệm L1.

Về độ chính xác, bảng có thể làm bạn ngạc nhiên. Với 10,000 ảnh huấn luyện và vài epoch, một SVM
trên đặc trưng HOG tốt có thể ngang hoặc hơn mạng nơ-ron. Đó chính là điều chương dặn: dựng một mốc
đơn giản trước khi dựng mạng nơ-ron, và chỉ giữ cách làm nơ-ron khi lợi ích đo được của nó xứng với
phần hệ thống nó thêm vào.

Thời gian mỗi ảnh cũng không tăng theo đúng tỉ lệ số phép. Phép nhân ma trận dày đặc chạy trên CPU
hiệu quả hơn nhiều so với phép atan2 hay phép cộng dồn rải rác vào histogram, nên HOG có thể chậm
hơn cả mạng dù ít phép hơn. Và cùng 109 phép so sánh, viết bằng vòng lặp Python thay vì phép toán
trên mảng, chậm hơn hẳn: cách hiện thực đổi chi phí nhiều không kém số phép.

## 4. Hai lời cảnh báo về quy mô

### Dựng mốc đơn giản trước

Một cạm bẫy chương nêu là dùng mạng nơ-ron cho bài toán giải được bằng cách đơn giản hơn: trên tập
dữ liệu nhỏ, hồi quy logistic có thể bằng hoặc hơn. Hồi quy logistic ở đây chính là mạng không có
tầng ẩn nào, 784 sang 10. Huấn luyện nó và mạng 784-128-64-10 trên ba cỡ tập huấn luyện, giữ số lần
cập nhật gần như nhau.

**Dự đoán:** với 100 ảnh huấn luyện, mô hình nào thắng? Với 10,000 ảnh thì sao?

In [ ]:
co_tap = [100, 1_000, 10_000]
ket_qua_moc = {"hồi quy logistic (784 → 10)": [], "mạng 784-128-64-10": []}
for n in co_tap:
    so_epoch = max(3, 30_000 // n)  # khoảng 1,000 lần cập nhật ở mọi cỡ tập
    for ten, rong in (("hồi quy logistic (784 → 10)", ()), ("mạng 784-128-64-10", (128, 64))):
        dat_hat_giong()
        m = mlp(rong)
        huan_luyen(m, X_hl[:n], y_hl[:n], so_epoch, F.cross_entropy, torch.optim.Adam(m.parameters(), lr=1e-3))
        ket_qua_moc[ten].append(100 * do_chinh_xac(m, X_kt, y_kt))
        do_tren_may(f"{n:>6,} ảnh, {ten}", ket_qua_moc[ten][-1], "%")

fig, ax = plt.subplots(figsize=(7, 3.8))
for (ten, kq), mau in zip(ket_qua_moc.items(), MAU[::2]):
    ax.semilogx(co_tap, kq, "o-", color=mau, lw=2, label=ten)
ax.set_xticks(co_tap, [f"{n:,}" for n in co_tap])
ax.set_xlabel("số ảnh huấn luyện")
ax.set_ylabel("độ chính xác trên tập kiểm tra (%)")
ax.set_title("Mốc đơn giản so với mạng nơ-ron, theo cỡ dữ liệu (đo trên máy này)")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

Hãy so khoảng cách giữa hai đường ở đầu trái và đầu phải. Nếu nó nhỏ khi dữ liệu ít và rộng ra khi
dữ liệu nhiều, thì mạng nơ-ron chỉ đáng phần chi phí của nó khi có đủ dữ liệu, đúng như chương nói.

### Lợi ích giảm dần

Chương còn nhắc rằng nới tầng ẩn từ 100 lên 1000 nơ-ron đưa số tham số từ khoảng 89.6 nghìn lên
khoảng 1.8 triệu, và mạng rộng hơn có chính xác hơn không thì phải huấn luyện mới biết. Mạng trong
phép tính đó có hai tầng ẩn cùng độ rộng, 784-h-h-10. Kiểm số tham số trước:

In [ ]:
def so_tham_so(rong):
    kich_thuoc = (784, *rong, 10)
    return sum(v * r + r for v, r in zip(kich_thuoc[:-1], kich_thuoc[1:]))


theo_sach("tham số của 784-100-100-10", so_tham_so((100, 100)), sach=89_600, nguon="~89,6 nghìn", tol=100)
theo_sach("tham số của 784-1000-1000-10", so_tham_so((1000, 1000)), sach=1.8e6, nguon="~1,8 triệu", tol=0.05e6)
theo_sach("FP32 của 784-100-100-10 (KB)", so_tham_so((100, 100)) * 4 / 1000, sach=358, nguon="358 KB · 7 MB")
theo_sach("FP32 của 784-1000-1000-10 (MB)", so_tham_so((1000, 1000)) * 4 / 1e6, sach=7, nguon="358 KB · 7 MB")
print(f"tham số tăng {so_tham_so((1000, 1000)) / so_tham_so((100, 100)):.1f} lần khi độ rộng tăng 10 lần")

Chương kể một cạm bẫy đi kèm: thấy độ chính xác tăng 5 điểm khi tham số tăng từ 10 nghìn lên 100
nghìn, nhóm phát triển ngoại suy thêm 5 điểm nữa ở 1 triệu. Ba độ rộng h = 16, 100 và 1000 dưới đây
cho khoảng 13 nghìn, 89.6 nghìn và 1.8 triệu tham số, tức gần đúng ba mốc đó.

**Dự đoán:** bước nhân tham số thứ hai, từ h = 100 lên h = 1000, đem lại nhiều điểm bằng bước thứ
nhất, hay ít hơn?

In [ ]:
cac_do_rong = [16, 32, 100, 300, 1000]
ket_qua_rong = []
for h in cac_do_rong:
    dat_hat_giong()
    m = mlp((h, h))
    t0 = time.perf_counter()
    huan_luyen(m, X_hl, y_hl, 3, F.cross_entropy, torch.optim.Adam(m.parameters(), lr=1e-3))
    ket_qua_rong.append((so_tham_so((h, h)), 100 * do_chinh_xac(m, X_kt, y_kt), time.perf_counter() - t0))
    do_tren_may(f"784-{h}-{h}-10: {ket_qua_rong[-1][0]:,} tham số, độ chính xác", ket_qua_rong[-1][1], "%")

tham, dung, giay = map(np.array, zip(*ket_qua_rong))
DIEM_MOI_BUOC = sach(5, nguon="5 điểm · 10 nghìn → 100 nghìn → 1 triệu")
a, b, c = (cac_do_rong.index(h) for h in (16, 100, 1000))
buoc_1, buoc_2 = dung[b] - dung[a], dung[c] - dung[b]
do_tren_may("bước thứ nhất, h = 16 → 100, độ chính xác tăng", buoc_1, "điểm")
do_tren_may("bước thứ hai, h = 100 → 1000, độ chính xác tăng", buoc_2, "điểm")
do_tren_may("thời gian huấn luyện, h = 1000 so với h = 100", giay[c] / giay[b], "lần")
print(f"cạm bẫy trong ví dụ của chương: thấy {DIEM_MOI_BUOC} điểm ở bước thứ nhất, chờ thêm {DIEM_MOI_BUOC} ở bước thứ hai")

# ngoại suy tuyến tính theo log tham số, đúng kiểu cạm bẫy: bước thứ hai lặp lại độ dốc của bước thứ nhất
doc = buoc_1 / np.log10(tham[b] / tham[a])
ngoai_suy = dung[b] + doc * np.log10(tham[c] / tham[b])
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.semilogx(tham, dung, "o-", color=MAU[0], lw=2, label="đo trên máy này")
ax.semilogx([tham[b], tham[c]], [dung[b], ngoai_suy], "--", color=MAU[1], lw=2,
            label="ngoại suy từ bước thứ nhất (cạm bẫy)")
ax.plot([tham[c]], [ngoai_suy], "o", mfc="none", mec=MAU[1], mew=2, ms=8)
for h, x, y in zip(cac_do_rong, tham, dung):
    ax.annotate(f"h = {h}", (x, y), xytext=(0, -16), textcoords="offset points", ha="center", fontsize=8)
ax.set_ylim(min(dung.min(), ngoai_suy) - 3, max(dung.max(), ngoai_suy) + 3)
ax.set_xlabel("số tham số (thang log)")
ax.set_ylabel("độ chính xác trên tập kiểm tra (%)")
ax.set_title("784-h-h-10, cùng 3 epoch trên 10,000 ảnh")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

Đường đo thường dốc ở bên trái rồi thoải dần ở bên phải, nên nằm dưới đường ngoại suy. Hai điểm kề
nhau có thể lên xuống thất thường, và mạng rộng nhất có khi còn kém mạng hẹp hơn nó một chút: với 3
epoch trên 10,000 ảnh, giới hạn nằm ở dữ liệu và thời gian huấn luyện chứ không ở độ rộng. Trong khi đó
bộ nhớ và thời gian huấn luyện thì tăng đều theo số tham số. Vì vậy chương dặn đo lợi ích biên trong
đúng chế độ của mình, thay vì ngoại suy tuyến tính.

Chương chỉ ra cùng điều đó trên bảng lịch sử của mình: lỗi ImageNet giảm từ mức của AlexNet xuống
mức của ResNet-152, trong khi FLOP huấn luyện tăng nhiều hơn hẳn.

In [ ]:
LOI_ALEXNET = sach(15.3, nguon="15,3 % → 3,6 %")
LOI_RESNET = sach(3.6, nguon="15,3 % → 3,6 %")
FLOP_ALEXNET = sach(5e17, nguon="5×10¹⁷ → ~10¹⁹ FLOP", tol=0.5e17)
FLOP_RESNET = sach(1e19, nguon="5×10¹⁷ → ~10¹⁹ FLOP", tol=0.5e19)
print(f"lỗi giảm {LOI_ALEXNET / LOI_RESNET:.2f} lần, FLOP huấn luyện tăng khoảng {FLOP_RESNET / FLOP_ALEXNET:.0f} lần")

## Thử thêm

1. Ở phần 1, đổi `NGUONG` thành 64 hoặc 200, hoặc tăng `MOI_CHU_SO` lên 30 (tức 300 phép so sánh).
   Độ chính xác của bộ quy tắc đổi bao nhiêu, và có đổi theo tỉ lệ số phép không?
2. Ở phần 2, đổi `NGAN` thành 4, hoặc `O_LUOI` thành 4 và `O_PIXEL` thành 7 (lưới 4×4 ô, mỗi ô
   7×7 pixel). Vectơ đặc trưng dài bao nhiêu, SVM tốn bao nhiêu phép, và độ chính xác ra sao? Dấu ✗
   ở dòng so với 441 khi đó là điều được chờ đợi.
3. Ở phần 4, thêm `h = 2000` vào `cac_do_rong`. Tham số, thời gian huấn luyện và độ chính xác tăng
   bao nhiêu so với `h = 1000`?

## Tóm lại

* Cùng một chữ số 28×28, ba lối tính tốn khoảng 100, khoảng 8,000 và 109,184 phép theo chương, tức
  gấp 80 rồi 1,091.8 lần; cách đếm của bạn có thể khác vài lần, nhưng không khác bậc độ lớn.
* Mỗi bước chuyển đổi cả dạng công việc: so sánh rẽ nhánh, phép toán có cấu trúc trên đặc trưng,
  rồi nhân ma trận dày đặc với tham số vượt bộ nhớ đệm L1.
* Nhiều phép hơn không tự động là chính xác hơn: một mốc đơn giản, đo trên đúng dữ liệu của bạn, là
  thứ phải dựng trước.
* Nới mạng thêm tham số thường đem lại ít điểm hơn sau mỗi lần nhân, trong khi chi phí thì tăng đều.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Tính toán nơ-ron](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch05-nn-computation/), dựng từ chương
[*Neural Computation*](https://mlsysbook.ai/vol1/nn_computation/nn_computation.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.